In [12]:
import pandas as pd
import re
import emoji

In [13]:
df_mx = pd.read_csv('../data/irosva.mx.training.csv')
print(f"Dataset cargado: {df_mx.shape}")
print(df_mx.head())

Dataset cargado: (2400, 4)
                                 ID           TOPIC  IS_IRONIC  \
0  6424ee0864a0af40660686e135f5652b  asuntosConacyt          1   
1  f59978451dd7fb228830fed2ae00c3ef  asuntosConacyt          1   
2  280963c5eb0d162858caf3480a7ea08c  asuntosConacyt          1   
3  69af1d02743953e5a0e971cf6ac00c69  asuntosConacyt          1   
4  b0be044f4682eea74c6fe83a87fff22a  asuntosConacyt          1   

                                             MESSAGE  
0  Rica económicamente, pero muy pobre en objetiv...  
1  En algo tiene razón, mafias hay en todo, hasta...  
2  ¿De cuándo acá tan preocupados por la ciencia ...  
3  De una vez que paren las titulaciones, que tod...  
4  @LopesDorigaa @Dolores_PL  Es el que también t...  


In [14]:
def preprocesar(texto):
    # 1. Reemplazar URLs
    texto = re.sub(r'http\S+|www\S+', '[URL]', texto)
    # 2. Reemplazar menciones @usuario
    texto = re.sub(r'@\w+', '[USER]', texto)
    # 3. Convertir emojis a texto en español
    texto = emoji.demojize(texto, language='es')
    # 4. Normalizar caracteres repetidos (máx 2)
    texto = re.sub(r'(.)\1{2,}', r'\1\1', texto)
    # 5. Eliminar caracteres extraños conservando puntuación relevante
    # Se agregan ¿ y ¡ porque son señales lingüísticas importantes para ironía
    texto = re.sub(r'[^\w\s\!\?\.\,\;\:\#\[\]áéíóúüñÁÉÍÓÚÜÑ¿¡]', ' ', texto)
    # 6. Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

In [15]:
df_mx['MESSAGE_CLEAN'] = df_mx['MESSAGE'].apply(preprocesar)
print("Preprocesamiento aplicado")

Preprocesamiento aplicado


In [16]:
for i in range(5):
    print("ORIGINAL: ", df_mx['MESSAGE'].iloc[i])
    print("LIMPIO:   ", df_mx['MESSAGE_CLEAN'].iloc[i])
    print()

ORIGINAL:  Rica económicamente, pero muy pobre en objetividad.
LIMPIO:    Rica económicamente, pero muy pobre en objetividad.

ORIGINAL:  En algo tiene razón, mafias hay en todo, hasta en su 4t.
LIMPIO:    En algo tiene razón, mafias hay en todo, hasta en su 4t.

ORIGINAL:  ¿De cuándo acá tan preocupados por la ciencia y la investigación?
LIMPIO:    ¿De cuándo acá tan preocupados por la ciencia y la investigación?

ORIGINAL:  De una vez que paren las titulaciones, que todos sean pasantes, así todos tendrán el mismo nivel que el director de Pemex y el subdirector de Conacyt 😡
LIMPIO:    De una vez que paren las titulaciones, que todos sean pasantes, así todos tendrán el mismo nivel que el director de Pemex y el subdirector de Conacyt :cara_cabreada:

ORIGINAL:  @LopesDorigaa @Dolores_PL  Es el que también te atiende tu tiendita.. a la Padierna...!!
LIMPIO:    [USER] [USER] Es el que también te atiende tu tiendita.. a la Padierna..!!



In [17]:
df_mx.to_csv('../data/irosva.mx.training.clean.csv', index=False)
print("Dataset guardado en data/irosva.mx.training.clean.csv")

Dataset guardado en data/irosva.mx.training.clean.csv
